# Accélérer pandas sur les ventes immobilières (DVF)

Ce notebook applique les techniques du chapitre « Accélération du calcul » à un vrai jeu de données volumineux : les ventes immobilières de France (DVF), 10,8 millions de ventes de 2014 à 2022. Chaque partie mesure ce qu'elle compare, pour que vous voyiez le gain :

1. Mémoire et chargement : mesurer, lire moins, choisir ses types
2. Boucles et vectorisation : la distance de chaque vente à Nantes
3. Regrouper vite, filtrer lisiblement : `agg` plutôt qu'`apply`, `query` et `eval`
4. Compiler avec Numba : `@njit`, `prange`, `engine="numba"`
5. Passer à l'échelle : Modin, Polars et DuckDB sur les neuf années

Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin. Les cellules qui suivent une solution affichent les vérifications et les temps : comparez-y vos résultats.

Le notebook tourne sans GPU, sur les 2 cœurs de Colab ou du Codespace, en quelques minutes. Les temps donnés dans les énoncés sont des ordres de grandeur, mesurés sur 2 cœurs : les vôtres varieront, les rapports entre méthodes beaucoup moins.

## Mise en place

La première cellule télécharge DVF depuis shuuchuu/datasets (635 Mo, un dossier Parquet par année) et installe Modin, que Colab n'a pas. La seconde importe les bibliothèques et définit trois outils de mesure :

- `mesurer(fonction, repetitions)` : le meilleur temps, en secondes, de plusieurs appels de `fonction()`, comme `%timeit`, mais en renvoyant le nombre ;
- `chronometrer(fonction)` : le résultat d'un seul appel, long, et son temps ;
- `tableau_temps(temps)` : un dictionnaire de temps en tableau, du plus lent au plus rapide, avec le gain sur le plus lent.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!pip install -q "modin[dask]"

In [ ]:
import time
import timeit

import duckdb
import numpy as np
import pandas as pd
import polars as pl
from geopy.distance import geodesic
from numba import njit, prange

CHEMIN = "dvf/dvf-linearized-2014-2022.parquet"
ANNEE_2022 = [("annee_mutation", "==", 2022)]


def mesurer(fonction, repetitions=3):
  """Le meilleur temps, en secondes, de `repetitions` appels de `fonction()`."""
  return min(timeit.repeat(fonction, number=1, repeat=repetitions))


def chronometrer(fonction):
  """Le résultat d'un appel de `fonction()`, et son temps en secondes."""
  debut = time.perf_counter()
  resultat = fonction()
  return resultat, time.perf_counter() - debut


def tableau_temps(temps):
  """Les temps (nom → secondes), du plus lent au plus rapide, et leur gain."""
  secondes = pd.Series(temps).sort_values(ascending=False)
  return pd.DataFrame(
    {
      "secondes": secondes.round(4),
      "× plus rapide que le plus lent": (secondes.max() / secondes).round(1),
    }
  )

## 1. Mémoire et chargement

DVF est un jeu Parquet partitionné : un dossier par année (`annee_mutation=2014/`…), que `pd.read_parquet` lit d'un bloc. Les parties 1 à 4 travaillent sur l'année 2022 (508 837 ventes), que `filters=` lit seule ; la partie 5, sur les neuf années.

In [ ]:
%time dvf = pd.read_parquet(CHEMIN, filters=ANNEE_2022)
dvf.shape

### 1.1 Mesurer la mémoire

Combien de mémoire occupe `dvf` ? Avec `memory_usage(deep=True)`, affichez la mémoire totale de la table en Mo (10⁶ octets), puis ses cinq colonnes les plus lourdes, avec leur type (`dvf.dtypes`). Comparez au total de `memory_usage()` sans `deep=True` : d'où vient l'écart ?

Vous devez trouver environ 206 Mo, dont 155 pour quatre colonnes de chaînes et l'index `id_mutation`, contre 72 Mo sans `deep=True`.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `dvf.memory_usage(deep=True)` renvoie une `Series` : le nombre d'octets de chaque colonne, plus une ligne `Index` ; sa somme divisée par `1e6` donne des Mo.
- `.nlargest(5)` garde ses cinq plus grandes valeurs ; `dvf.dtypes.reindex(...)` donne le type de ces colonnes (`NaN` pour l'`Index`, qui n'en est pas une).
- Sans `deep=True`, une colonne `object` compte 8 octets par ligne : un pointeur vers chaque chaîne, pas la chaîne elle-même.

Un squelette, où chaque `...` est à compléter :

```python
memoire = dvf.memory_usage(deep=True)
print(f"Avec deep=True : {memoire.sum() / 1e6:.0f} Mo")
sans_deep = ...
print(f"Sans deep=True : {sans_deep / 1e6:.0f} Mo")
lourdes = memoire.nlargest(5)
pd.DataFrame({"Mo": (lourdes / 1e6).round(1), "type": ...})
```

#### Solution

In [ ]:
memoire = dvf.memory_usage(deep=True)
print(f"Avec deep=True : {memoire.sum() / 1e6:.0f} Mo")
sans_deep = dvf.memory_usage().sum()
print(f"Sans deep=True : {sans_deep / 1e6:.0f} Mo")
# Les chaînes en object : un objet Python par valeur, que seul deep=True compte.
lourdes = memoire.nlargest(5)
pd.DataFrame(
  {"Mo": (lourdes / 1e6).round(1), "type": dvf.dtypes.reindex(lourdes.index)}
)

### 1.2 Lire moins

La suite du notebook n'utilise que les douze colonnes de `COLONNES` (cellule ci-dessous). Relisez 2022 avec `columns=COLONNES` en plus de `filters=`, dans une variable `dvf` qui remplace la précédente, et mesurez la lecture avec `%time` : comparez-la à celle de la table entière. Affichez la mémoire de la nouvelle table.

L'index `id_mutation`, des chaînes, est relu même avec `columns=`. Retirez-le avec `reset_index(drop=True)`, qui le remplace par un `RangeIndex` (des entiers calculés à la demande, sans mémoire), et affichez de nouveau la mémoire.

Vous devez trouver une lecture environ deux fois plus rapide, 85 Mo, puis 55 Mo sans l'index.

**Documentation utile**

- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html

In [ ]:
COLONNES = [
  "mois_mutation",
  "jour_mutation",
  "nature_mutation",
  "valeur_fonciere",
  "code_postal",
  "nom_commune",
  "code_departement",
  "surface_reelle_bati_total",
  "nombre_maisons",
  "nombre_appartements",
  "latitude",
  "longitude",
]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `%time dvf = ...` mesure une affectation et en garde le résultat.
- `filters=ANNEE_2022` comme plus haut ; `columns=` prend une liste de noms de colonnes.
- `drop=True` jette l'index au lieu d'en faire une colonne.

Un squelette, où chaque `...` est à compléter :

```python
%time dvf = pd.read_parquet(CHEMIN, columns=..., filters=...)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
dvf = ...
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo sans l'index")
```

#### Solution

In [ ]:
%time dvf = pd.read_parquet(CHEMIN, columns=COLONNES, filters=ANNEE_2022)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
dvf = dvf.reset_index(drop=True)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo sans l'index")

### 1.3 Choisir ses types

Comparez la mémoire de la colonne `nom_commune` (`memory_usage(deep=True)` d'une `Series`) dans trois types : `object`, son type actuel, `string[pyarrow]` et `category`. Puis, dans `dvf`, convertissez `nom_commune` en `category`, et `code_postal` (`int64`) avec `pd.to_numeric(..., downcast="unsigned")`, et affichez la mémoire totale de `dvf`. `nature_mutation` et `code_departement` sont déjà des `category` : Parquet les stocke en dictionnaire, et pandas les relit ainsi.

Vous devez trouver environ 33, 10 et 4 Mo pour `nom_commune`, `uint32` pour `code_postal`, et 24 Mo en tout : plus de huit fois moins que la table du départ.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `dvf["nom_commune"].astype("string[pyarrow]")` convertit sans modifier `dvf` ; `.memory_usage(deep=True)` donne ses octets.
- `pd.to_numeric(serie, downcast="unsigned")` choisit le plus petit entier non signé qui garde toutes les valeurs.
- Pour modifier `dvf`, affectez les colonnes converties : `dvf["nom_commune"] = ...`.

Un squelette, où chaque `...` est à compléter :

```python
communes = dvf["nom_commune"]
for type_ in ["object", "string[pyarrow]", "category"]:
  memoire = communes.astype(type_).memory_usage(deep=True)
  print(f"nom_commune en {type_} : {memoire / 1e6:.1f} Mo")
dvf["nom_commune"] = ...
dvf["code_postal"] = ...
print(dvf["code_postal"].dtype)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
```

#### Solution

In [ ]:
communes = dvf["nom_commune"]
for type_ in ["object", "string[pyarrow]", "category"]:
  memoire = communes.astype(type_).memory_usage(deep=True)
  print(f"nom_commune en {type_} : {memoire / 1e6:.1f} Mo")
dvf["nom_commune"] = communes.astype("category")
dvf["code_postal"] = pd.to_numeric(dvf["code_postal"], downcast="unsigned")
print(dvf["code_postal"].dtype)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")

La table telle que la suite l'utilise, avec ses types et sa mémoire :

In [ ]:
dvf.info(memory_usage="deep")

## 2. Boucles et vectorisation

Quelle distance sépare chaque vente du centre de Nantes ? `geopy.distance.geodesic` la calcule exactement, sur l'ellipsoïde terrestre, mais un point à la fois, en Python : de l'ordre d'un dixième de milliseconde par point. La cellule suivante s'en sert comme référence sur 2 000 ventes de Loire-Atlantique.

In [ ]:
%%time
NANTES = 47.218371, -1.553621  # latitude, longitude du centre de Nantes
loire = dvf[dvf["code_departement"] == "44"]  # Loire-Atlantique
echantillon = loire.head(2000)
reference = np.array(
  [
    geodesic(NANTES, (lat, lon)).km
    for lat, lon in zip(echantillon["latitude"], echantillon["longitude"])
  ]
)

### 2.1 La formule de haversine en NumPy

Sur une sphère de rayon $R$ = 6 371 km, la formule de haversine donne la distance entre deux points de latitudes $\varphi_1$, $\varphi_2$ et de longitudes $\lambda_1$, $\lambda_2$, en radians :

$$d = 2R \arcsin \sqrt{\sin^2 \frac{\varphi_2 - \varphi_1}{2} + \cos \varphi_1 \cos \varphi_2 \sin^2 \frac{\lambda_2 - \lambda_1}{2}}$$

Écrivez `haversine(lat1, lon1, lat2, lon2)`, qui prend des degrés et renvoie des km, avec les seules fonctions de NumPy (`np.radians`, `np.sin`, `np.cos`, `np.arcsin`, `np.sqrt`) : elle marche alors aussi bien sur deux points que sur des colonnes entières, sans boucle. La cellule qui suit la solution la compare à `geodesic` sur les 2 000 ventes d'`echantillon`.

Vous devez trouver `haversine(48.8566, 2.3522, *NANTES)` ≈ 343 km (Paris–Nantes), et un écart relatif à `geodesic` inférieur à 0,5 % : la Terre n'est pas tout à fait une sphère.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.arcsin.html

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
  """La distance en km entre (lat1, lon1) et (lat2, lon2), donnés en degrés."""
  ...  # Votre code ici

#### Conseils supplémentaires

- Convertissez d'abord les quatre angles en radians, un `np.radians` par variable.
- $\sin^2 x$ s'écrit `np.sin(x) ** 2`.
- Sans `if` ni boucle, la même fonction accepte des nombres, des tableaux NumPy et des `Series` : les opérations de NumPy s'appliquent élément par élément, avec le broadcasting entre un point et une colonne.

Un squelette, où chaque `...` est à compléter :

```python
def haversine(lat1, lon1, lat2, lon2):
  lat1, lon1 = np.radians(lat1), np.radians(lon1)
  lat2, lon2 = ...
  a = np.sin((lat2 - lat1) / 2) ** 2 + ...
  return 2 * 6371 * ...
```

#### Solution

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
  """La distance en km entre (lat1, lon1) et (lat2, lon2), donnés en degrés."""
  lat1, lon1 = np.radians(lat1), np.radians(lon1)
  lat2, lon2 = np.radians(lat2), np.radians(lon2)
  a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
  )
  return 2 * 6371 * np.arcsin(np.sqrt(a))

In [ ]:
print(f"Paris–Nantes : {haversine(48.8566, 2.3522, *NANTES):.0f} km")
distances = haversine(*NANTES, echantillon["latitude"], echantillon["longitude"])
ecart = np.abs(distances - reference) / reference
print(f"Écart relatif maximal à geodesic : {ecart.max():.2%}")

### 2.2 Cinq façons de boucler, et aucune

Calculez la distance à Nantes de chaque vente de `loire` (18 260 ventes), avec `haversine` appelée sur une vente à la fois, de cinq façons :

- une compréhension de liste sur `loire.iterrows()` ;
- `loire.apply(..., axis=1)` ;
- une compréhension sur `loire.itertuples()` ;
- une compréhension sur `zip(loire["latitude"], loire["longitude"])` ;
- `np.vectorize(haversine)` ;

puis par un seul appel de `haversine` sur les colonnes entières. Mesurez chacune avec `mesurer(lambda: ..., repetitions=1)`, rangée dans le dictionnaire `temps` sous le nom de sa méthode ; la cellule qui suit la solution les affiche en tableau.

Enfin, calculez avec la version vectorisée la distance à Nantes de toutes les ventes de 2022, mesurée avec `%timeit`, et comptez celles à moins de 10 km.

Vous devez trouver la version vectorisée plus de cent fois plus rapide que la meilleure boucle, `np.vectorize` à peine plus rapide que `zip`, toute la France en moins d'un dixième de seconde, et environ 6 350 ventes à moins de 10 km de Nantes.

In [ ]:
temps = {}
# Votre code ici

#### Conseils supplémentaires

- Une boucle à mesurer va dans une `lambda` sans argument : `mesurer(lambda: [... for _, vente in loire.iterrows()], repetitions=1)`. Une seule répétition suffit pour ces boucles, longues ; la version vectorisée, rapide, garde les 3 répétitions par défaut.
- `iterrows` donne des couples (index, ligne) ; `apply(axis=1)` passe chaque ligne, une `Series`, à sa fonction ; `itertuples` donne des tuples nommés (`vente.latitude`).
- `np.vectorize(haversine)` renvoie une fonction qui boucle sur ses arguments : appelez-la comme `haversine`.
- `*NANTES` passe la latitude et la longitude de Nantes comme deux arguments.
- `%timeit distances = ...` ne garde pas `distances` : recalculez-la sur la ligne suivante pour compter, avec `(distances < 10).sum()`.

Un squelette, où chaque `...` est à compléter :

```python
temps = {}
temps["iterrows"] = mesurer(
  lambda: [
    haversine(*NANTES, vente["latitude"], vente["longitude"])
    for _, vente in loire.iterrows()
  ],
  repetitions=1,
)
temps["apply(axis=1)"] = mesurer(lambda: loire.apply(..., axis=1), repetitions=1)
temps["itertuples"] = ...
temps["zip"] = ...
temps["np.vectorize"] = ...
temps["vectorisée"] = mesurer(
  lambda: haversine(*NANTES, loire["latitude"], loire["longitude"])
)

%timeit haversine(*NANTES, dvf["latitude"], dvf["longitude"])
distances = ...
print((distances < 10).sum(), "ventes à moins de 10 km de Nantes")
```

#### Solution

In [ ]:
temps = {}
temps["iterrows"] = mesurer(
  lambda: [
    haversine(*NANTES, vente["latitude"], vente["longitude"])
    for _, vente in loire.iterrows()
  ],
  repetitions=1,
)
temps["apply(axis=1)"] = mesurer(
  lambda: loire.apply(
    lambda vente: haversine(*NANTES, vente["latitude"], vente["longitude"]), axis=1
  ),
  repetitions=1,
)
temps["itertuples"] = mesurer(
  lambda: [
    haversine(*NANTES, vente.latitude, vente.longitude) for vente in loire.itertuples()
  ],
  repetitions=1,
)
temps["zip"] = mesurer(
  lambda: [
    haversine(*NANTES, lat, lon)
    for lat, lon in zip(loire["latitude"], loire["longitude"])
  ],
  repetitions=1,
)
temps["np.vectorize"] = mesurer(
  lambda: np.vectorize(haversine)(*NANTES, loire["latitude"], loire["longitude"]),
  repetitions=1,
)
temps["vectorisée"] = mesurer(
  lambda: haversine(*NANTES, loire["latitude"], loire["longitude"])
)

%timeit haversine(*NANTES, dvf["latitude"], dvf["longitude"])
distances = haversine(*NANTES, dvf["latitude"], dvf["longitude"])
print((distances < 10).sum(), "ventes à moins de 10 km de Nantes")

In [ ]:
tableau_temps(temps)

Les cinq boucles appellent `haversine` une fois par vente, en Python, avec des nombres : chaque appel coûte autant que le calcul vectorisé sur des milliers de ventes. `iterrows` et `apply(axis=1)` construisent en plus une `Series` par ligne. `np.vectorize` n'est qu'une boucle Python plus commode.

## 3. Regrouper vite, filtrer lisiblement

Le prix au m² d'une commune : la somme des valeurs foncières de ses ventes de bâti, divisée par la somme de leurs surfaces. Une commune se repère par son département et son nom, car des communes de départements différents portent le même. La version naïve passe chaque groupe à une fonction Python avec `apply` ; elle prend plusieurs secondes :

In [ ]:
bati = dvf[(dvf["valeur_fonciere"] > 0) & (dvf["surface_reelle_bati_total"] > 0)]
communes = bati.groupby(["code_departement", "nom_commune"], observed=True)


def prix_m2_commune(ventes):
  return ventes["valeur_fonciere"].sum() / ventes["surface_reelle_bati_total"].sum()


%time naif = communes.apply(prix_m2_commune, include_groups=False)
naif.head()

### 3.1 Agréger, puis diviser

Calculez le même prix au m² sans `apply`. Une agrégation nommée de `communes` (`agg`) donne en une fois, par commune, la somme des valeurs (colonne `valeur`), celle des surfaces (`surface`) et le nombre de ventes (`ventes`), dans un DataFrame `sommes` ; la division de deux de ses colonnes donne le prix, `prix_m2`. Mesurez la cellule avec `%%time`, sur sa première ligne. La cellule qui suit la solution vérifie que `prix_m2` est égal à `naif`, et affiche les communes les plus chères parmi celles d'au moins 100 ventes.

Vous devez trouver un calcul des centaines de fois plus rapide, les mêmes prix, et des arrondissements de Paris en tête.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une agrégation nommée prend un couple (colonne, fonction) par colonne du résultat : `agg(valeur=("valeur_fonciere", "sum"), ...)`.
- `"size"` compte les lignes de chaque groupe.
- `"sum"` est une fonction intégrée, en C : un groupe n'est jamais confié à Python.

Un squelette, où chaque `...` est à compléter :

```python
%%time
sommes = communes.agg(
  valeur=("valeur_fonciere", "sum"),
  surface=...,
  ventes=...,
)
prix_m2 = ...
```

#### Solution

In [ ]:
%%time
sommes = communes.agg(
  valeur=("valeur_fonciere", "sum"),
  surface=("surface_reelle_bati_total", "sum"),
  ventes=("valeur_fonciere", "size"),
)
prix_m2 = sommes["valeur"] / sommes["surface"]

In [ ]:
print("Mêmes prix que la version naïve :", np.allclose(prix_m2, naif))
prix_m2[sommes["ventes"] >= 100].nlargest(10).round()

### 3.2 `query` et `eval`

Les ventes d'appartements seuls (sans maison) se filtrent par un masque, dans la cellule ci-dessous, qui mesure aussi ce filtre et le prix au m² de ces ventes.

Écrivez le même filtre avec `dvf.query(...)`, dans une fonction `appartements_query()`, et le prix au m² d'`appartements` avec `appartements.eval(...)`, puis mesurez-les avec `%timeit` : comparez aux deux mesures de la cellule ci-dessous. Affichez le nombre de ventes que garde chaque filtre, et le prix au m² médian d'un appartement.

Vous devez trouver 159 376 ventes par les deux filtres, `query` un peu plus lent que le masque et `eval` une dizaine de fois plus lent que la division : leur intérêt est la lisibilité, pas la vitesse.

In [ ]:
def appartements_masque():
  return dvf[
    (dvf["nombre_appartements"] > 0)
    & (dvf["nombre_maisons"] == 0)
    & (dvf["valeur_fonciere"] > 0)
    & (dvf["surface_reelle_bati_total"] > 0)
  ]


appartements = appartements_masque()
%timeit appartements_masque()
%timeit appartements["valeur_fonciere"] / appartements["surface_reelle_bati_total"]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Dans le texte d'une requête, les colonnes se nomment sans `dvf[...]`, et `and` remplace `&` ; deux chaînes côte à côte, sur deux lignes, n'en font qu'une.
- `eval` prend une expression sur les colonnes, et renvoie une `Series`.

Un squelette, où chaque `...` est à compléter :

```python
def appartements_query():
  return dvf.query("nombre_appartements > 0 and nombre_maisons == 0 and ...")


%timeit appartements_query()
%timeit appartements.eval(...)
print(len(appartements_query()), len(appartements), "ventes")
prix = appartements.eval(...)
print(f"Prix médian d'un appartement : {prix.median():.0f} €/m²")
```

#### Solution

In [ ]:
def appartements_query():
  return dvf.query(
    "nombre_appartements > 0 and nombre_maisons == 0"
    " and valeur_fonciere > 0 and surface_reelle_bati_total > 0"
  )


%timeit appartements_query()
%timeit appartements.eval("valeur_fonciere / surface_reelle_bati_total")
print(len(appartements_query()), len(appartements), "ventes")
prix = appartements.eval("valeur_fonciere / surface_reelle_bati_total")
print(f"Prix médian d'un appartement : {prix.median():.0f} €/m²")

## 4. Compiler avec Numba

À quelle distance de la plus proche des dix plus grandes villes de France se trouve chaque vente ? Il faut, pour chacune des 508 837 ventes, ses dix distances aux villes, puis la plus petite. En NumPy, le broadcasting d'une colonne `(n, 1)` contre une ligne `(10,)` calcule les dix distances de chaque vente d'un coup, dans un tableau temporaire `(n, 10)` : c'est la référence, mesurée ci-dessous.

`haversine_njit` est votre `haversine` compilée par Numba (`njit(haversine)`), pour être appelée depuis une autre fonction Numba.

In [ ]:
VILLES = pd.DataFrame(
  [
    ("Paris", 48.8566, 2.3522),
    ("Marseille", 43.2965, 5.3698),
    ("Lyon", 45.7640, 4.8357),
    ("Toulouse", 43.6047, 1.4442),
    ("Nice", 43.7102, 7.2620),
    ("Nantes", 47.2184, -1.5536),
    ("Montpellier", 43.6108, 3.8767),
    ("Strasbourg", 48.5734, 7.7521),
    ("Bordeaux", 44.8378, -0.5792),
    ("Lille", 50.6292, 3.0573),
  ],
  columns=["ville", "latitude", "longitude"],
)
lat, lon = dvf["latitude"].to_numpy(), dvf["longitude"].to_numpy()
lat_v, lon_v = VILLES["latitude"].to_numpy(), VILLES["longitude"].to_numpy()
haversine_njit = njit(haversine)


def distance_ville_proche_numpy(lat, lon, lat_v, lon_v):
  distances = haversine(lat[:, np.newaxis], lon[:, np.newaxis], lat_v, lon_v)
  return distances.min(axis=1)  # la plus petite des 10 distances de chaque vente


temps_numba = {
  "NumPy": mesurer(lambda: distance_ville_proche_numpy(lat, lon, lat_v, lon_v))
}
temps_numba

### 4.1 `@njit`, puis `parallel=True`

Numba compile une fonction Python qui boucle sur des tableaux NumPy. Écrivez `distance_ville_proche(lat, lon, lat_v, lon_v)`, décorée par `@njit`, avec deux boucles : pour chaque vente `i`, la plus petite des distances `haversine_njit(lat[i], lon[i], lat_v[j], lon_v[j])` aux villes `j`, sans tableau temporaire.

Puis écrivez `distance_ville_proche_parallele`, la même fonction décorée par `@njit(parallel=True)`, dont la boucle sur les ventes passe de `range` à `prange` : ses itérations, indépendantes, se répartissent sur les cœurs.

Mesurez le premier appel de chacune avec `%time` : il compile la fonction. Puis ajoutez leur temps, `mesurer(lambda: ...)`, à `temps_numba`, sous les noms `"@njit"` et `"@njit(parallel=True)"`. La cellule qui suit la solution vérifie leurs résultats et affiche les temps.

Vous devez trouver une compilation de l'ordre d'une seconde, `@njit` un peu plus rapide que NumPy, qui remplit puis parcourt son tableau temporaire, et `parallel=True` jusqu'à deux fois plus rapide encore sur 2 cœurs.

In [ ]:
@njit
def distance_ville_proche(lat, lon, lat_v, lon_v): ...  # Votre code ici


@njit(parallel=True)
def distance_ville_proche_parallele(lat, lon, lat_v, lon_v): ...  # Votre code ici

#### Conseils supplémentaires

- `np.empty(len(lat))` réserve le tableau du résultat ; `np.inf` est un bon point de départ pour une plus petite valeur, que `min(a, b)` met à jour.
- Dans une fonction `@njit`, une boucle sur des nombres est compilée, aussi rapide qu'en C : c'est ce qui permet de se passer du tableau `(n, 10)`.
- `prange` vient de `numba`, importé au début du notebook ; seule la boucle extérieure devient `prange`.
- Une `TypingError` de Numba vient souvent d'un argument qui n'est pas un tableau NumPy (une `Series`, par exemple) : passez `lat`, `lon`, `lat_v` et `lon_v`.
- `%time proche = ...` : sans affectation, `%time` afficherait tout le tableau.

Un squelette, où chaque `...` est à compléter :

```python
@njit
def distance_ville_proche(lat, lon, lat_v, lon_v):
  proche = np.empty(len(lat))
  for i in range(len(lat)):
    plus_petite = np.inf
    for j in range(len(lat_v)):
      plus_petite = min(plus_petite, ...)
    proche[i] = plus_petite
  return proche


@njit(parallel=True)
def distance_ville_proche_parallele(lat, lon, lat_v, lon_v): ...


%time proche = distance_ville_proche(lat, lon, lat_v, lon_v)
%time proche = distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
temps_numba["@njit"] = mesurer(lambda: distance_ville_proche(lat, lon, lat_v, lon_v))
temps_numba["@njit(parallel=True)"] = ...
```

#### Solution

In [ ]:
@njit
def distance_ville_proche(lat, lon, lat_v, lon_v):
  proche = np.empty(len(lat))
  for i in range(len(lat)):
    plus_petite = np.inf
    for j in range(len(lat_v)):
      plus_petite = min(plus_petite, haversine_njit(lat[i], lon[i], lat_v[j], lon_v[j]))
    proche[i] = plus_petite
  return proche


@njit(parallel=True)
def distance_ville_proche_parallele(lat, lon, lat_v, lon_v):
  proche = np.empty(len(lat))
  for i in prange(len(lat)):  # itérations indépendantes, réparties sur les cœurs
    plus_petite = np.inf
    for j in range(len(lat_v)):
      plus_petite = min(plus_petite, haversine_njit(lat[i], lon[i], lat_v[j], lon_v[j]))
    proche[i] = plus_petite
  return proche


%time proche = distance_ville_proche(lat, lon, lat_v, lon_v)
%time proche = distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
temps_numba["@njit"] = mesurer(lambda: distance_ville_proche(lat, lon, lat_v, lon_v))
temps_numba["@njit(parallel=True)"] = mesurer(
  lambda: distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
)

In [ ]:
proche = distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
reference_numpy = distance_ville_proche_numpy(lat, lon, lat_v, lon_v)
print("Mêmes distances qu'en NumPy :", np.allclose(proche, reference_numpy))
print(f"{(proche < 20).mean():.0%} des ventes à moins de 20 km d'une des dix villes")
tableau_temps(temps_numba)

### 4.2 `engine="numba"` : une médiane glissante

`rolling(...).apply(f, raw=True)` appelle `f` sur chaque fenêtre, un tableau NumPy, en Python ; avec `engine="numba"`, pandas compile `f` et la boucle sur les fenêtres. La cellule ci-dessous donne le prix au m² des ventes parisiennes de 2022, dans l'ordre de leurs dates, avec un `RangeIndex` (`ignore_index=True`).

Calculez la médiane glissante de `prix_paris` sur 500 ventes de trois façons, et rangez leur temps dans `temps_glissant` :

- `rolling(500).apply(np.median, raw=True)` (une seule répétition) ;
- la même avec `engine="numba"` : mesurez son premier appel, la compilation, avec `%time`, avant `mesurer` ;
- la méthode intégrée, `rolling(500).median()`.

La cellule qui suit la solution vérifie qu'elles donnent les mêmes médianes.

Vous devez trouver `engine="numba"` environ quatre fois plus rapide que Python, une fois compilé, mais `median()` encore dix fois plus rapide : une méthode intégrée bat Numba, quand elle existe.

In [ ]:
paris = bati[bati["code_departement"] == "75"].sort_values(
  ["mois_mutation", "jour_mutation"], ignore_index=True
)
prix_paris = paris["valeur_fonciere"] / paris["surface_reelle_bati_total"]
len(prix_paris)

In [ ]:
temps_glissant = {}
# Votre code ici

#### Conseils supplémentaires

- `fenetres = prix_paris.rolling(500)` se réutilise pour les trois calculs.
- `raw=True` passe chaque fenêtre en tableau NumPy, ce qu'exige `engine="numba"` : Numba ne sait pas compiler une `Series`.
- Le premier appel avec `engine="numba"` compile ; les suivants réutilisent le code compilé, gardé en cache par pandas.

Un squelette, où chaque `...` est à compléter :

```python
temps_glissant = {}
fenetres = prix_paris.rolling(500)
temps_glissant["Python"] = mesurer(
  lambda: fenetres.apply(np.median, raw=True), repetitions=1
)
%time mediane = fenetres.apply(...)
temps_glissant["Numba"] = mesurer(lambda: ...)
temps_glissant["median()"] = ...
```

#### Solution

In [ ]:
temps_glissant = {}
fenetres = prix_paris.rolling(500)
temps_glissant["Python"] = mesurer(
  lambda: fenetres.apply(np.median, raw=True), repetitions=1
)
%time mediane = fenetres.apply(np.median, raw=True, engine="numba")
temps_glissant["Numba"] = mesurer(
  lambda: fenetres.apply(np.median, raw=True, engine="numba")
)
temps_glissant["median()"] = mesurer(lambda: fenetres.median())

In [ ]:
mediane = prix_paris.rolling(500).median()
mediane_numba = prix_paris.rolling(500).apply(np.median, raw=True, engine="numba")
print("Mêmes médianes :", np.allclose(mediane, mediane_numba, equal_nan=True))
tableau_temps(temps_glissant)

### 4.3 `engine="numba"` dans un `groupby`

L'écart interquartile (IQR) du prix au m² de chaque commune, `np.percentile(x, 75) - np.percentile(x, 25)` sur les prix de ses ventes, mesure leur dispersion. La cellule ci-dessous groupe les prix par commune. Calculez l'IQR de trois façons, et rangez leur temps dans `temps_iqr` :

- `agg` d'une `lambda` Python (une seule répétition : plusieurs secondes) ;
- `agg(iqr_groupe, engine="numba")`, où `iqr_groupe(values, index)` a la signature qu'impose pandas : mesurez son premier appel, la compilation, avec `%time` ;
- `quantile(0.75) - quantile(0.25)`, deux méthodes intégrées.

Vous devez trouver Numba plus de cent fois plus rapide que la `lambda`, et même plus rapide que les deux `quantile`, qui trient chaque groupe deux fois : ici, seule la mesure départage Numba et la méthode intégrée.

In [ ]:
prix_bati = bati["valeur_fonciere"] / bati["surface_reelle_bati_total"]
prix_communes = prix_bati.groupby(
  [bati["code_departement"], bati["nom_commune"]], observed=True
)

In [ ]:
temps_iqr = {}
# Votre code ici

#### Conseils supplémentaires

- Avec `engine="numba"`, pandas passe à la fonction les valeurs du groupe et son index, deux tableaux NumPy : `def iqr_groupe(values, index)`, même si elle n'utilise pas `index`.
- `engine="numba"` lève une `TypingError` quand l'index de la `Series` est fait de chaînes : celui de `prix_bati` vient du `RangeIndex` de la partie 1 (`reset_index(drop=True)`).
- `prix_communes.quantile(0.75)` donne une `Series` indexée par commune, comme `agg`.

Un squelette, où chaque `...` est à compléter :

```python
def iqr_groupe(values, index):
  return ...


temps_iqr = {}
temps_iqr["Python"] = mesurer(lambda: prix_communes.agg(lambda x: ...), repetitions=1)
%time iqr = prix_communes.agg(iqr_groupe, engine="numba")
temps_iqr["Numba"] = ...
temps_iqr["quantile"] = mesurer(lambda: ...)
```

#### Solution

In [ ]:
def iqr_groupe(values, index):
  return np.percentile(values, 75) - np.percentile(values, 25)


temps_iqr = {}
temps_iqr["Python"] = mesurer(
  lambda: prix_communes.agg(lambda x: np.percentile(x, 75) - np.percentile(x, 25)),
  repetitions=1,
)
%time iqr = prix_communes.agg(iqr_groupe, engine="numba")
temps_iqr["Numba"] = mesurer(lambda: prix_communes.agg(iqr_groupe, engine="numba"))
temps_iqr["quantile"] = mesurer(
  lambda: prix_communes.quantile(0.75) - prix_communes.quantile(0.25)
)

In [ ]:
iqr_quantile = prix_communes.quantile(0.75) - prix_communes.quantile(0.25)
print("Mêmes IQR :", np.allclose(iqr, iqr_quantile))
tableau_temps(temps_iqr)

## 5. Passer à l'échelle : 2014–2022

Le prix au m² médian de chaque département, sur les neuf années : 10,8 millions de ventes. pandas lit les trois colonnes utiles, et l'index `id_mutation` avec elles, puis filtre et groupe : une dizaine de secondes ou plus, et plus de 2 Go de mémoire au plus fort. `par_departement` met un résultat sous une forme commune (une `Series` indexée par le code du département, en texte, triée), pour comparer ceux des quatre bibliothèques.

In [ ]:
def prix_departements_pandas():
  dvf = pd.read_parquet(
    CHEMIN,
    columns=["code_departement", "valeur_fonciere", "surface_reelle_bati_total"],
  )
  bati = dvf.query("surface_reelle_bati_total > 0")
  prix = bati["valeur_fonciere"] / bati["surface_reelle_bati_total"]
  return prix.groupby(bati["code_departement"], observed=True).median()


def par_departement(prix):
  """`prix` en Series indexée par le code du département, en texte, triée."""
  return pd.Series(prix.to_numpy(), index=prix.index.astype(str)).sort_index()


temps_echelle = {}
prix_pandas, temps_echelle["pandas"] = chronometrer(prix_departements_pandas)
print(f"pandas : {temps_echelle['pandas']:.1f} s")
prix_pandas.nlargest(5).round()

### 5.1 Modin

Modin reprend l'API de pandas sur plusieurs cœurs : il suffit d'importer `modin.pandas` à la place de `pandas`. Écrivez `prix_departements_modin()`, la fonction `prix_departements_pandas` avec `mpd.read_parquet` (`import modin.pandas as mpd`), qui renvoie un résultat pandas (`.modin.to_pandas()`). Mesurez-la avec `chronometrer`, dans `prix_modin` et `temps_echelle["Modin"]`.

Modin démarre d'abord ses processus (moteur Dask), et prévient qu'il lit ce Parquet partitionné avec pandas (« Defaulting to pandas ») avant de répartir la table : c'est normal. Vous devez trouver Modin plus lent que pandas sur 2 cœurs : ses processus se partagent les mêmes cœurs, et la table leur est transférée.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `import modin.pandas as mpd` : `mpd.read_parquet` prend les mêmes arguments que `pd.read_parquet`, et le DataFrame qu'il renvoie, les mêmes méthodes.
- Le résultat d'une opération de Modin est un objet de Modin : `.modin.to_pandas()` le convertit.

Un squelette, où chaque `...` est à compléter :

```python
import modin.pandas as mpd


def prix_departements_modin():
  dvf = mpd.read_parquet(...)
  bati = ...
  prix = ...
  return prix.groupby(...).median().modin.to_pandas()


prix_modin, temps_echelle["Modin"] = chronometrer(prix_departements_modin)
print(f"Modin : {temps_echelle['Modin']:.1f} s")
```

#### Solution

In [ ]:
import modin.pandas as mpd


def prix_departements_modin():
  dvf = mpd.read_parquet(
    CHEMIN,
    columns=["code_departement", "valeur_fonciere", "surface_reelle_bati_total"],
  )
  bati = dvf.query("surface_reelle_bati_total > 0")
  prix = bati["valeur_fonciere"] / bati["surface_reelle_bati_total"]
  medianes = prix.groupby(bati["code_departement"], observed=True).median()
  return medianes.modin.to_pandas()


prix_modin, temps_echelle["Modin"] = chronometrer(prix_departements_modin)
print(f"Modin : {temps_echelle['Modin']:.1f} s")

### 5.2 Polars

Écrivez `prix_departements_polars()`, le même calcul en Polars : une requête paresseuse, `pl.scan_parquet` sur tous les fichiers (`f"{CHEMIN}/**/*.parquet"`), `filter`, `group_by`, `agg` de la médiane du prix au m², nommée `prix_m2` (`alias`), puis `collect()`, qui l'exécute. Renvoyez-en une `Series` pandas indexée par `code_departement` (`.to_pandas()`, `set_index`). Mesurez-la avec `chronometrer`, dans `prix_polars` et `temps_echelle["Polars"]`.

Vous devez trouver Polars plus de dix fois plus rapide que pandas : il ne lit que les colonnes et les lignes utiles, sur tous les cœurs.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `pl.col("nom")` désigne une colonne dans une expression ; les expressions se combinent comme des colonnes pandas (`/`, `>`) et ont leurs méthodes (`.median()`, `.alias("nom")`).
- Rien n'est lu avant `collect()` : Polars optimise d'abord tout le plan.
- Le DataFrame Polars n'a pas d'index : `.to_pandas().set_index("code_departement")["prix_m2"]`.

Un squelette, où chaque `...` est à compléter :

```python
def prix_departements_polars():
  prix = (
    pl.scan_parquet(f"{CHEMIN}/**/*.parquet")
    .filter(...)
    .group_by("code_departement")
    .agg(
      (pl.col("valeur_fonciere") / pl.col("surface_reelle_bati_total"))
      .median()
      .alias("prix_m2")
    )
    .collect()
  )
  return ...


prix_polars, temps_echelle["Polars"] = chronometrer(prix_departements_polars)
print(f"Polars : {temps_echelle['Polars']:.1f} s")
```

#### Solution

In [ ]:
def prix_departements_polars():
  prix = (
    pl.scan_parquet(f"{CHEMIN}/**/*.parquet")
    .filter(pl.col("surface_reelle_bati_total") > 0)
    .group_by("code_departement")
    .agg(
      (pl.col("valeur_fonciere") / pl.col("surface_reelle_bati_total"))
      .median()
      .alias("prix_m2")
    )
    .collect()
  )
  return prix.to_pandas().set_index("code_departement")["prix_m2"]


prix_polars, temps_echelle["Polars"] = chronometrer(prix_departements_polars)
print(f"Polars : {temps_echelle['Polars']:.1f} s")

### 5.3 DuckDB

Écrivez `prix_departements_duckdb()`, le même calcul en SQL : `duckdb.sql(...)` sur les fichiers Parquet (`FROM '...'`, avec le même motif `**/*.parquet`), `median(...)` du prix au m², `WHERE`, `GROUP BY`, puis `.df()` pour un DataFrame pandas, dont vous renvoyez une `Series` indexée par `code_departement`. Mesurez-la avec `chronometrer`, dans `prix_duckdb` et `temps_echelle["DuckDB"]`. La cellule qui suit la solution compare les résultats et les temps des quatre bibliothèques.

Vous devez trouver DuckDB au moins aussi rapide que Polars, et les quatre résultats égaux.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une f-string insère `CHEMIN` dans la requête ; le chemin, dans la requête, est entre apostrophes.
- `AS prix_m2` nomme la colonne du résultat.

Un squelette, où chaque `...` est à compléter :

```python
def prix_departements_duckdb():
  prix = duckdb.sql(f"""
    SELECT code_departement,
           median(...) AS prix_m2
    FROM '{CHEMIN}/**/*.parquet'
    WHERE ...
    GROUP BY code_departement
  """).df()
  return ...


prix_duckdb, temps_echelle["DuckDB"] = chronometrer(prix_departements_duckdb)
print(f"DuckDB : {temps_echelle['DuckDB']:.1f} s")
```

#### Solution

In [ ]:
def prix_departements_duckdb():
  prix = duckdb.sql(f"""
    SELECT code_departement,
           median(valeur_fonciere / surface_reelle_bati_total) AS prix_m2
    FROM '{CHEMIN}/**/*.parquet'
    WHERE surface_reelle_bati_total > 0
    GROUP BY code_departement
  """).df()
  return prix.set_index("code_departement")["prix_m2"]


prix_duckdb, temps_echelle["DuckDB"] = chronometrer(prix_departements_duckdb)
print(f"DuckDB : {temps_echelle['DuckDB']:.1f} s")

In [ ]:
reference = par_departement(prix_pandas)
for nom, prix in [
  ("Modin", prix_modin),
  ("Polars", prix_polars),
  ("DuckDB", prix_duckdb),
]:
  egal = np.allclose(par_departement(prix), reference)
  print(f"{nom} donne le même résultat que pandas : {egal}")
tableau_temps(temps_echelle)

## À retenir

- Mesurer d'abord : `%timeit`, `%time`, `memory_usage(deep=True)`.
- Lire moins (`columns=`, `filters=`) et choisir ses types (`category`, `string[pyarrow]`, `downcast`) : souvent le premier gain, et le plus simple.
- Vectoriser : une opération sur des colonnes entières plutôt qu'une boucle, `apply` ou `np.vectorize`, qui appellent Python à chaque ligne.
- Grouper avec des fonctions intégrées (`agg` de `"sum"`, `"size"`…), pas avec `apply` ; `query` et `eval` servent la lisibilité.
- Numba, quand une boucle ne se vectorise pas, ou mal : `@njit`, `prange`, `engine="numba"`, et une mesure contre la méthode intégrée.
- Au-delà de la mémoire, ou pour aller plus vite : Polars et DuckDB plutôt que Modin, sur une machine à peu de cœurs.